## 🤖 可以使用的 prompt（可自行修改）

**① 請 AI 寫規格書初稿**
```
我正在考期中考，要做一個「系學會活動報名系統」（Google Colab + SQLite + Groq + Gradio）。
請依照下面的題目與規格範本，幫我寫出規格書初稿。測試清單請遵守數量上限：AI 解讀測試 8～10 句、流程測試 8～10 步。
【題目】
（貼上題目）
【規格範本】
（貼上規格範本）
```

**② 請 AI 依規格寫程式**
```
你是 Python 助教。請依照下面的規格，幫我寫 Google Colab 程式，並遵守以下要求：
1. 第 1 格已經存在，請從第 2 格開始，依序為：(2) 建立資料庫與初始資料 (3) 顯示資料表 (4) AI 解讀函式 (5) AI 解讀測試 (6) 功能函式 (7) 主流程與流程測試 (8) Gradio 網頁。每格分開給我，開頭用註解說明這格在做什麼。
2. 第 1 格已經 import 了 os、re、time、sqlite3、pandas as pd、display、gradio as gr，並定義了 client、MODEL、REASONING_EFFORT、IN_COLAB，請直接使用，不要重新讀取金鑰，也不要把金鑰寫在程式裡。
3. 呼叫 Groq 時使用 model=MODEL、temperature=0、reasoning_effort=REASONING_EFFORT，並用 try/except 包起來；遇到超過使用量的錯誤（429）時，等待 30 秒後重試，最多 3 次；每次呼叫後 time.sleep(2)；同一句話不要重複呼叫 AI（可以用 dict 暫存結果）。
4. 每次執行都要先 DROP TABLE IF EXISTS，再重建資料表並放入初始資料。
5. 所有 SQL 都用 ? 參數，不可以用 f-string 組 SQL。
6. 不可直接相信 AI 的回覆：要從回覆中找出允許清單裡的值，找不到就當作「未知」；數量用正規表示式找出數字，找不到就當作 1；數量是否在允許範圍內由程式檢查。
7. fetchone() 之後一定要先檢查是否為 None；資料庫連線用 try/finally 確保關閉。
8. 統計查詢要使用 JOIN 與 GROUP BY。
9. 第 5 格的測試格式為 (使用者說的話, 預期動作, 預期對象, 預期數量)，"*" 表示不在意；跑完要印出 AI 原始回覆、系統解讀，以及通過幾句。
10. 第 8 格啟動網頁時寫成：if IN_COLAB: demo.launch(share=True)。
11. 只用第 1 格已 import 的套件。註解用繁體中文，寫得讓初學者看得懂。

規格如下：
（貼上你的規格書）
```

**③ 出錯時請 AI 協助除錯**
```
我的 Colab 程式出現以下錯誤。請先用兩三句話解釋原因，再告訴我「只需要修改哪幾行」，不要整份重寫。
【錯誤訊息】
（貼上完整錯誤訊息）
【出錯的那一格程式】
（貼上程式）
```

# 📋 系學會活動報名系統規格書
學號：411322469 姓名：洪唯皓 卷別：A

## 1. 系統目的
讓學生用一句話報名、取消或查詢活動，並讓幹部查看每場活動的名額統計。

## 2. 使用者會怎麼說
1. 我和兩個朋友想報名程式工作坊
2. 程式工作坊還有幾個名額？
3. 我要取消程式工作坊
4. 我要參加迎新晚會
5. 我要參加籃球友誼賽
6. 各活動報名狀況

## 3. 資料表
|表名|欄位|型態|說明|
|---|---|---|---|
|events|title|TEXT|活動名稱，主鍵|
|events|day|TEXT|活動日期|
|events|capacity|INTEGER|總名額|
|registrations|id|INTEGER|流水號，主鍵|
|registrations|student|TEXT|報名者姓名|
|registrations|title|TEXT|對應活動名稱的外鍵|
|registrations|people|INTEGER|本次報名總人數，含本人|
|registrations|status|TEXT|已報名或已取消|

每次執行第 2 格重建兩張表，報名紀錄從空表開始。

|活動|日期|名額|
|---|---|---|
|迎新晚會|2026-11-20|6|
|程式工作坊|2026-11-25|4|
|籃球友誼賽|2026-11-28|3|

## 4. AI 的工作
將一句中文轉成 `動作,活動,人數`。動作允許報名、取消、查詢、統計；活動只能是上表的三個名稱，統計時用「全部」。辨識不到的欄位用「未知」。沒提人數預設 1；「我和兩個朋友」合計 3 人。AI 不查資料庫、不決定能否報名。Python 驗證解讀結果，遇到不明需求就列出活動供選擇。

## 5. 功能與規則
|功能|條件|系統回應|資料庫變化|
|---|---|---|---|
|報名|有姓名、人數 1～3、同活動無有效報名、目前報名未達 2 場、剩餘名額足夠|回覆人數與剩餘名額|新增已報名紀錄|
|取消|姓名與活動有有效紀錄|確認取消|改為已取消，釋出整筆人數|
|查詢|指定活動，可不填姓名|總名額減有效報名人數加總|無|
|統計|要求各活動報名狀況|依日期列出每場活動、總名額、已報名人數、剩餘名額，含零報名活動|無|

## 6. 例外情況
1. 同一姓名重複報名同一活動會被拒絕；取消後可重新報名。
2. 剩餘 1 人但要報名 2 人，整筆拒絕。
3. 報名或取消缺少姓名時提示填寫；查無有效紀錄時提示無法取消。
4. 報名 0 人或超過 3 人時拒絕；已有兩場有效報名時拒絕第三場。
5. 遇到無關問題或未知活動，列出現有活動，不更動紀錄。

## 7. 測試清單
### 7-1 AI 解讀測試
|#|類型|使用者說的話|預期動作|預期活動|預期人數|
|---|---|---|---|---|---|
|1|正常|我和兩個朋友想報名程式工作坊|報名|程式工作坊|3|
|2|正常|程式工作坊還有幾個名額？|查詢|程式工作坊|1|
|3|正常|我要取消程式工作坊|取消|程式工作坊|1|
|4|正常|我要參加迎新晚會|報名|迎新晚會|1|
|5|正常|我要參加籃球友誼賽|報名|籃球友誼賽|1|
|6|正常|各活動報名狀況|統計|全部|1|
|7|錯誤|我要報名游泳課|報名|未知|1|
|8|錯誤|明天會下雨嗎？|未知|未知|1|
|9|邊界|我要報名籃球友誼賽，共4人|報名|籃球友誼賽|4|
|10|邊界|幫我報名程式工作坊，總共2人|報名|程式工作坊|2|

### 7-2 流程測試
從初始資料依序操作；「空白」表示不填姓名。

|#|姓名|使用者說的話|預期結果|
|---|---|---|---|
|1|小安|我和兩個朋友想報名程式工作坊|報名成功，程式工作坊共 3 人，剩餘 1 人。|
|2|小安|我和兩個朋友想報名程式工作坊|已報名此活動，請勿重複報名。|
|3|小林|幫我報名程式工作坊，總共2人|名額不足，剩餘 1 人。|
|4|空白|程式工作坊還有幾個名額？|程式工作坊剩餘 1 人。|
|5|小安|我要參加迎新晚會|報名成功，迎新晚會共 1 人，剩餘 5 人。|
|6|小安|我要參加籃球友誼賽|每人最多同時報名 2 個活動。|
|7|小安|我要取消程式工作坊|已取消報名。|
|8|空白|程式工作坊還有幾個名額？|程式工作坊剩餘 4 人。|
|9|小安|我要取消程式工作坊|找不到有效報名紀錄。|
|10|空白|各活動報名狀況|統計：三場活動已報名人數依日期為 1、0、0，剩餘 5、4、3。|

In [12]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [13]:
# 第 2 格：重建活動與報名紀錄
DB = "club_events.sqlite"
EVENTS = [("迎新晚會", "2026-11-20", 6), ("程式工作坊", "2026-11-25", 4),
          ("籃球友誼賽", "2026-11-28", 3)]
def reset_db():
    db = sqlite3.connect(DB)
    try:
        with db:
            db.execute("BEGIN IMMEDIATE")
            db.execute("DROP TABLE IF EXISTS registrations")
            db.execute("DROP TABLE IF EXISTS events")
            db.execute("CREATE TABLE events (title TEXT PRIMARY KEY, day TEXT NOT NULL, capacity INTEGER NOT NULL)")
            db.execute("""CREATE TABLE registrations (
                id INTEGER PRIMARY KEY, student TEXT NOT NULL,
                title TEXT NOT NULL REFERENCES events(title),
                people INTEGER NOT NULL CHECK(people BETWEEN 1 AND 3),
                status TEXT NOT NULL CHECK(status IN ('已報名','已取消')))""")
            db.execute("CREATE UNIQUE INDEX active_entry ON registrations(student,title) WHERE status='已報名'")
            db.executemany("INSERT INTO events VALUES (?,?,?)", EVENTS)
    finally:
        db.close()
reset_db()


In [14]:
# 第 3 格：查看活動與所有歷史報名紀錄
def show_tables():
    db = sqlite3.connect(DB)
    try:
        print("活動")
        display(pd.read_sql_query("SELECT * FROM events ORDER BY day", db))
        print("報名紀錄")
        display(pd.read_sql_query("SELECT * FROM registrations ORDER BY id", db))
    finally:
        db.close()
show_tables()


活動


,title,day,capacity
0,迎新晚會,2026-11-20,6
1,程式工作坊,2026-11-25,4
2,籃球友誼賽,2026-11-28,3


報名紀錄


,id,student,title,people,status


In [15]:
# 第 4 格：AI 只解讀需求，資料庫操作由 Python 負責
ACTIONS = {"報名", "取消", "查詢", "統計"}
TITLES = {event[0] for event in EVENTS}
cache = {}
instructions = """把使用者的需求轉成一行：動作,活動,人數。只輸出這三欄。
動作限報名、取消、查詢、統計、未知。活動限迎新晚會、程式工作坊、籃球友誼賽。
查剩餘名額是查詢；查各活動報名狀況是統計，活動填全部。無法確認的欄位填未知。
明確的報名動作即使活動不存在仍輸出報名,未知,人數。無關問題輸出未知,未知,1。
人數用阿拉伯數字，未提到用1。我和兩位朋友是3人。保留0、負數或超過3的值，交給程式檢查。"""

def interpret(sentence):
    sentence = (sentence or "").strip()
    if not sentence:
        return "未知", "未知", 1, ""
    if sentence in cache:
        return cache[sentence]
    for attempt in range(4):  # 第一次呼叫加上最多三次重試
        try:
            result = client.chat.completions.create(
                model=MODEL, temperature=0, reasoning_effort=REASONING_EFFORT,
                messages=[{"role":"system", "content":instructions},
                          {"role":"user", "content":sentence}])
            raw = (result.choices[0].message.content or "").strip()
            break
        except Exception as error:
            if getattr(error, "status_code", None) == 429 and attempt < 3:
                time.sleep(30)
            else:
                return "服務錯誤", "未知", 1, "Groq 暫時無法回應，請檢查金鑰或額度後重試。"
        finally:
            time.sleep(2)
    fields = [v.strip() for v in re.split("[,，]", raw)]
    action, title, people = "未知", "未知", 1
    if len(fields) == 3:
        action = fields[0] if fields[0] in ACTIONS else "未知"
        title = fields[1] if fields[1] in TITLES else "未知"
        if action == "統計" and fields[1] == "全部":
            title = "全部"
        number = re.fullmatch(r"([+-]?\d+)\s*人?", fields[2])
        if number:
            people = int(number.group(1))
        elif re.search(r"\d", fields[2]):
            action = "未知"  # 有數字但格式不明，例如 1.5 人
    cache[sentence] = action, title, people, raw
    return cache[sentence]


In [16]:
# 第 5 格：將實際模型回覆與規格逐句比對
ai_tests = [
    ('我和兩個朋友想報名程式工作坊', '報名', '程式工作坊', 3),
    ('程式工作坊還有幾個名額？', '查詢', '程式工作坊', 1),
    ('我要取消程式工作坊', '取消', '程式工作坊', 1),
    ('我要參加迎新晚會', '報名', '迎新晚會', 1),
    ('我要參加籃球友誼賽', '報名', '籃球友誼賽', 1),
    ('各活動報名狀況', '統計', '全部', 1),
    ('我要報名游泳課', '報名', '未知', 1),
    ('明天會下雨嗎？', '未知', '未知', 1),
    ('我要報名籃球友誼賽，共4人', '報名', '籃球友誼賽', 4),
    ('幫我報名程式工作坊，總共2人', '報名', '程式工作坊', 2),
]
passed = 0
for sentence, action, title, people in ai_tests:
    a, t, p, raw = interpret(sentence)
    ok = all(w == "*" or w == v for w,v in zip((action,title,people),(a,t,p)))
    passed += ok
    print("✓" if ok else "✗", sentence, "｜原始回覆：", raw,
          "｜系統解讀：", (a,t,p), "｜預期：", (action,title,people))
print(f"AI 解讀通過 {passed}/{len(ai_tests)} 句")


✓ 我和兩個朋友想報名程式工作坊 ｜原始回覆： 報名,程式工作坊,3 ｜系統解讀： ('報名', '程式工作坊', 3) ｜預期： ('報名', '程式工作坊', 3)
✓ 程式工作坊還有幾個名額？ ｜原始回覆： 查詢,程式工作坊,1 ｜系統解讀： ('查詢', '程式工作坊', 1) ｜預期： ('查詢', '程式工作坊', 1)
✓ 我要取消程式工作坊 ｜原始回覆： 取消,程式工作坊,1 ｜系統解讀： ('取消', '程式工作坊', 1) ｜預期： ('取消', '程式工作坊', 1)
✓ 我要參加迎新晚會 ｜原始回覆： 報名,迎新晚會,1 ｜系統解讀： ('報名', '迎新晚會', 1) ｜預期： ('報名', '迎新晚會', 1)
✓ 我要參加籃球友誼賽 ｜原始回覆： 報名,籃球友誼賽,1 ｜系統解讀： ('報名', '籃球友誼賽', 1) ｜預期： ('報名', '籃球友誼賽', 1)
✓ 各活動報名狀況 ｜原始回覆： 統計,全部,1 ｜系統解讀： ('統計', '全部', 1) ｜預期： ('統計', '全部', 1)
✓ 我要報名游泳課 ｜原始回覆： 報名,未知,1 ｜系統解讀： ('報名', '未知', 1) ｜預期： ('報名', '未知', 1)
✓ 明天會下雨嗎？ ｜原始回覆： 未知,未知,1 ｜系統解讀： ('未知', '未知', 1) ｜預期： ('未知', '未知', 1)
✓ 我要報名籃球友誼賽，共4人 ｜原始回覆： 報名,籃球友誼賽,4 ｜系統解讀： ('報名', '籃球友誼賽', 4) ｜預期： ('報名', '籃球友誼賽', 4)
✓ 幫我報名程式工作坊，總共2人 ｜原始回覆： 報名,程式工作坊,2 ｜系統解讀： ('報名', '程式工作坊', 2) ｜預期： ('報名', '程式工作坊', 2)
AI 解讀通過 10/10 句


In [17]:
# 第 6 格：統計與報名規則
summary_sql = """SELECT e.title AS 活動, e.day AS 日期, e.capacity AS 名額,
    COALESCE(SUM(r.people),0) AS 已報名人數,
    e.capacity-COALESCE(SUM(r.people),0) AS 剩餘名額
    FROM events e LEFT JOIN registrations r
    ON e.title=r.title AND r.status='已報名'
    GROUP BY e.title,e.day,e.capacity ORDER BY e.day"""

def apply_request(action, title, people, student):
    student = (student or "").strip()
    if action not in ACTIONS or (action != "統計" and title not in TITLES):
        return "目前活動：" + "、".join(item[0] for item in EVENTS)
    if action in ("報名", "取消") and not student:
        return "請填寫姓名。"
    if action == "報名" and not 1 <= people <= 3:
        return "報名人數限 1～3 人。"
    db = sqlite3.connect(DB)
    try:
        with db:
            if action == "統計":
                return pd.read_sql_query(summary_sql, db).to_string(index=False)
            if action in ("報名", "取消"):
                db.execute("BEGIN IMMEDIATE")  # 讓名額檢查與寫入在同一筆交易中完成
            if action == "取消":
                changed = db.execute("""UPDATE registrations SET status='已取消'
                    WHERE student=? AND title=? AND status='已報名'""", (student,title))
                return "已取消報名。" if changed.rowcount else "找不到有效報名紀錄。"
            row = db.execute("SELECT 剩餘名額 FROM (" + summary_sql + ") WHERE 活動=?", (title,)).fetchone()
            if row is None:
                return "找不到活動。"
            remaining = row[0]
            if action == "查詢":
                return f"{title}剩餘 {remaining} 人。"
            active = db.execute("""SELECT title FROM registrations
                WHERE student=? AND status='已報名'""", (student,)).fetchall()
            if (title,) in active:
                return "已報名此活動，請勿重複報名。"
            if len(active) >= 2:
                return "每人最多同時報名 2 個活動。"
            if people > remaining:
                return f"名額不足，剩餘 {remaining} 人。"
            db.execute("INSERT INTO registrations(student,title,people,status) VALUES (?,?,?,'已報名')",
                       (student,title,people))
            return f"報名成功，{title}共 {people} 人，剩餘 {remaining-people} 人。"
    finally:
        db.close()


In [18]:
# 第 7 格：讓測試與網頁共用同一個入口
def handle(sentence, student=""):
    action, title, people, raw = interpret(sentence)
    if action == "服務錯誤":
        return raw
    return apply_request(action, title, people, student)

# 從空白報名紀錄開始，讓這一格可以獨立重跑
reset_db()
flow_tests = [
    ('小安', '我和兩個朋友想報名程式工作坊', '報名成功，程式工作坊共 3 人，剩餘 1 人。'),
    ('小安', '我和兩個朋友想報名程式工作坊', '已報名此活動，請勿重複報名。'),
    ('小林', '幫我報名程式工作坊，總共2人', '名額不足，剩餘 1 人。'),
    ('', '程式工作坊還有幾個名額？', '程式工作坊剩餘 1 人。'),
    ('小安', '我要參加迎新晚會', '報名成功，迎新晚會共 1 人，剩餘 5 人。'),
    ('小安', '我要參加籃球友誼賽', '每人最多同時報名 2 個活動。'),
    ('小安', '我要取消程式工作坊', '已取消報名。'),
    ('', '程式工作坊還有幾個名額？', '程式工作坊剩餘 4 人。'),
    ('小安', '我要取消程式工作坊', '找不到有效報名紀錄。'),
    ('', '各活動報名狀況', '統計：三場活動已報名人數依日期為 1、0、0，剩餘 5、4、3。'),
]
passed = 0
for i, (student, sentence, expected) in enumerate(flow_tests, 1):
    actual = handle(sentence, student)
    if sentence == "各活動報名狀況":
        # 比對標題與每列數字，也驗證沒有報名的活動被保留
        rows = [line.split() for line in actual.splitlines()]
        ok = rows == [["活動","日期","名額","已報名人數","剩餘名額"],
                      ["迎新晚會","2026-11-20","6","1","5"],
                      ["程式工作坊","2026-11-25","4","0","4"],
                      ["籃球友誼賽","2026-11-28","3","0","3"]]
    else:
        ok = actual == expected
    passed += ok
    print(i, "✓" if ok else "✗", sentence, "｜預期：", expected, "｜實際：", actual)
print(f"流程測試通過 {passed}/{len(flow_tests)} 步")
show_tables()


1 ✓ 我和兩個朋友想報名程式工作坊 ｜預期： 報名成功，程式工作坊共 3 人，剩餘 1 人。 ｜實際： 報名成功，程式工作坊共 3 人，剩餘 1 人。
2 ✓ 我和兩個朋友想報名程式工作坊 ｜預期： 已報名此活動，請勿重複報名。 ｜實際： 已報名此活動，請勿重複報名。
3 ✓ 幫我報名程式工作坊，總共2人 ｜預期： 名額不足，剩餘 1 人。 ｜實際： 名額不足，剩餘 1 人。
4 ✓ 程式工作坊還有幾個名額？ ｜預期： 程式工作坊剩餘 1 人。 ｜實際： 程式工作坊剩餘 1 人。
5 ✓ 我要參加迎新晚會 ｜預期： 報名成功，迎新晚會共 1 人，剩餘 5 人。 ｜實際： 報名成功，迎新晚會共 1 人，剩餘 5 人。
6 ✓ 我要參加籃球友誼賽 ｜預期： 每人最多同時報名 2 個活動。 ｜實際： 每人最多同時報名 2 個活動。
7 ✓ 我要取消程式工作坊 ｜預期： 已取消報名。 ｜實際： 已取消報名。
8 ✓ 程式工作坊還有幾個名額？ ｜預期： 程式工作坊剩餘 4 人。 ｜實際： 程式工作坊剩餘 4 人。
9 ✓ 我要取消程式工作坊 ｜預期： 找不到有效報名紀錄。 ｜實際： 找不到有效報名紀錄。
10 ✓ 各活動報名狀況 ｜預期： 統計：三場活動已報名人數依日期為 1、0、0，剩餘 5、4、3。 ｜實際：    活動         日期  名額  已報名人數  剩餘名額
 迎新晚會 2026-11-20   6      1     5
程式工作坊 2026-11-25   4      0     4
籃球友誼賽 2026-11-28   3      0     3
流程測試通過 10/10 步
活動


,title,day,capacity
0,迎新晚會,2026-11-20,6
1,程式工作坊,2026-11-25,4
2,籃球友誼賽,2026-11-28,3


報名紀錄


,id,student,title,people,status
0,1,小安,程式工作坊,3,已取消
1,2,小安,迎新晚會,1,已報名


In [19]:
# 第 8 格：建立網頁介面
demo = gr.Interface(fn=handle, inputs=[gr.Textbox(label="需求", placeholder="程式工作坊還有幾個名額？"),
    gr.Textbox(label="姓名（報名與取消必填）")], outputs=gr.Textbox(label="系統回應"),
    title="系學會活動報名", description="可以報名、取消、查詢名額，或查看各活動報名狀況。")
if IN_COLAB:
    demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df3a8857ad1909c880.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
